# MTP evaluation on Kaggle

Runs `scripts/evaluate.py` on published runs: per-head eval + self-speculative decoding, written as INTERFACES §10 JSON.

**Settings (right sidebar):** Accelerator **GPU T4 x2**, Internet **On**.
**Secrets** (Add-ons → Secrets): `HF_TOKEN` from an HF account that has accepted
https://huggingface.co/datasets/facebook/flores (without it only the IndicCorp datasets work).
**Inputs:** *Add Input →* the runs to evaluate: the `mtp-run-<ID>` datasets **or** the training notebook's output,
not both. Every attached folder with a `config.yaml` is one run.

**How to run (speculative decoding takes a while):**
1. Run cells 1-4 interactively. After a minute, cell 4 prints the first lines of each log; check that each run is on its own GPU.
2. Stop the session (power icon), then **Save Version → Save & Run All (Commit)**. A commit keeps running with
   the browser closed (up to 12 h); an interactive session stops when the tab sits idle.
3. When the version finishes, open its **Output** tab and download `results/` (and `eval_*.log` if something failed).

Runs are spread over the GPUs (run i on GPU i mod #GPUs), and each GPU works through its runs one after another.
Each dataset's JSON is written as soon as it finishes, so a partial run still leaves usable files.

In [ ]:
# 1. parameters
DATASETS = ["indiccorp_eval", "flores_hi"]   # also: indiccorp_eval_small, flores_mr
GROUPER = ""                                  # empty = data.grouper from each run's config
STEP = ""                                     # empty = latest step
DUMP_TOKENS = False                           # per-token dump for Jai's error analysis
SPEC_DECODE = True                            # self-speculative decoding (runs with 1 head are skipped)
POLICIES = ["fixed_k", "confidence_cut"]      # + "group_aware" once JN-9 lands
SPEC_N = 200                                  # prompts per dataset (first 8-16 words of a sentence)
MAX_NEW_TOKENS = 64
BRANCH = "main"

In [ ]:
# 2. secrets, clone, install
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set")
except Exception:
    print("no HF_TOKEN secret: FLORES will fail (gated); IndicCorp still works")

%cd /kaggle/working
!rm -rf /tmp/mtp && git clone -q --depth 1 --branch {BRANCH} https://github.com/rampuriajainam/mtp-indic-wordgroup.git /tmp/mtp
%cd /tmp/mtp
!git log -1 --oneline
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10; peft 0.21 refuses LoRA when it is present
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 3. find attached runs and the GPUs
import glob, subprocess
RUNS = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.yaml", recursive=True)})
N_GPUS = max(1, len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines()))
print(f"{N_GPUS} GPU(s)")
print("\n".join(RUNS) or "no run attached: Add Input -> mtp-run-<ID>")

In [ ]:
# 4. start the evaluation in the background: one queue per GPU, runs on a GPU one after another
import shlex, time
args = ["--datasets", *DATASETS, "--out", "/kaggle/working/results"]
if GROUPER: args += ["--grouper", GROUPER]
if STEP: args += ["--step", str(STEP)]
if DUMP_TOKENS: args += ["--dump_tokens"]
if SPEC_DECODE: args += ["--spec_decode", "--spec_n", str(SPEC_N), "--max_new_tokens", str(MAX_NEW_TOKENS), "--policies", *POLICIES]

queues = {g: [] for g in range(N_GPUS)}
for i, run in enumerate(RUNS):
    queues[i % N_GPUS].append(run)
lines = ["cd /tmp/mtp", "export PYTHONUTF8=1"]
for g, runs in queues.items():
    cmds = []
    for run in runs:
        log = f"/kaggle/working/eval_{os.path.basename(run.rstrip('/'))}.log"
        cmds.append(f"echo 'start {run} on GPU {g}'; CUDA_VISIBLE_DEVICES={g} python -u scripts/evaluate.py "
                    f"--run_dir {shlex.quote(run)} {' '.join(shlex.quote(a) for a in args)} > {log} 2>&1; "
                    f"echo 'done {run} (exit '$?')'")
    if cmds:
        lines.append("(" + "; ".join(cmds) + ") &")
lines.append("wait")
open("/tmp/eval_all.sh", "w").write("\n".join(lines) + "\n")
print(open("/tmp/eval_all.sh").read())

proc = subprocess.Popen(["bash", "/tmp/eval_all.sh"], stdout=open("/kaggle/working/eval_all.out", "w"),
                        stderr=subprocess.STDOUT, start_new_session=True)
time.sleep(90)
!cat /kaggle/working/eval_all.out
!tail -n 4 /kaggle/working/eval_*.log

In [ ]:
# 5. wait until every run is evaluated (keeps a committed session alive)
print("waiting for the evaluation to finish ...", flush=True)
print("eval_all.sh exit code", proc.wait())
!cat /kaggle/working/eval_all.out
!grep -h -i -B2 -A8 "error\|traceback" /kaggle/working/eval_*.log | tail -n 40

In [ ]:
# 6. show the JSON
import json
for path in sorted(glob.glob("/kaggle/working/results/*/eval_*.json")):
    rec = json.load(open(path, encoding="utf-8"))
    print(f"\n{rec['run_name']} | {rec['dataset']} | step {rec['step']} | grouper {rec['grouper']}")
    for r in rec["per_head"] or []:
        ig, bd = r["top1_in_group"], r["top1_at_boundary"]
        print(f"  head {r['head']}: loss {r['loss']:.3f}  top1 {100*r['top1']:.1f}%  top5 {100*r['top5']:.1f}%"
              + (f"  in-group {100*ig:.1f}% / boundary {100*bd:.1f}%" if ig is not None and bd is not None else ""))
    for e in rec.get("spec_decode") or []:
        gi = e["group_integrity"]
        print(f"  spec {e['policy']}: accepted len {e['mean_accepted_len']:.2f}, speed-up x{e['speedup']:.2f}"
              f" ({e['tokens_per_sec']:.1f} vs {e['greedy_tokens_per_sec']:.1f} tok/s), match greedy {e['outputs_match_greedy']}"
              + (f", group integrity {100*gi:.0f}%" if gi is not None else ""))